# 02 · 웨이크 워드 → 제스처 → 모델 교체 + cam1 리모컨

**목적** — 01번 노트북 두 개와 `gesture_live.py` 를 한 파이프라인으로 묶고,
인식된 제스처로 DPU 에 올라간 모델과 cam1 의 화면을 조종한다.

```
        [항상]  마이크 ──── openWakeWord (CPU) ─────┐
        [항상]  cam0 ─┐                            │  event_q
        [항상]  cam1 ─┴─ 캡처 스레드 (최신 프레임)   ▼
                                            ┌─────────────┐
                                            │  DPU 워커   │  ← DPU 는 하나뿐이다
                                            └──────┬──────┘
                                                   │ 제스처 이름을 같은 큐로
                                                   ▼
                                          apply_event(state, ev)
                                          ├─ dpu: 어느 xmodel 을 올릴지
                                          └─ cx/cy/zoom: cam1 디지털 팬·줌
```

| 이벤트 | 동작 |
|---|---|
| 웨이크 워드 | 두 화면을 켜고 cam0 에 **제스처 xmodel** 을 올린다 (`dpu=gesture`) |
| `left` `right` `up` `down` | cam1 뷰포트를 좌·우·상·하로 옮긴다 |
| `thumbs up` / `thumbs down` | cam1 을 줌 인 / 줌 아웃 |
| `close` | cam0 의 xmodel 을 끈다. DPU 유휴 (`dpu=idle`) |
| `open` | cam1 의 **두 번째 xmodel** 을 켠다. cam0 제스처는 꺼진다 (`dpu=task`) |
| `bye` | 두 화면을 정지시킨다 |
| `hi` | 무동작 |

**핵심 제약은 DPU 가 하나라는 것이다.** 두 xmodel 을 동시에 올릴 수 없으므로
모드 전환이 곧 `overlay.load_model()` 호출이다. 비트스트림은 시작할 때 한 번만 올린다.

그래서 `close`/`open`/`bye` 를 하면 **제스처 인식이 멈춘다.** cam0 의 xmodel 이 내려갔으니
당연한 결과다. 그 상태에서 `left` 나 `open` 은 잡히지 않는다 — 돌아오는 길은 웨이크
워드뿐이고, 그게 웨이크 워드를 CPU(ONNX)에 두는 이유다. DPU 가 뭘 하든 계속 듣는다.

**두 번째 xmodel 은 지금 제스처 xmodel 을 그대로 다시 올린 대역(stand-in)이다.**
교체 동작·지연·복귀는 전부 진짜다. 진짜 2번 모델이 생기면 `TASK_XMODEL` 한 줄만 바꾼다.

cam1 의 팬·줌은 프레임을 잘라 확대하는 디지털 방식이다 (C270 에 하드웨어 PTZ 컨트롤이
없다). 모델을 갈아 끼워도 뷰포트는 그대로 남는다.

전제: `00_dual_camera_check` PASS, `01_wakeword` PASS, `gesture_live.py` 단독 실행 확인.

## 1. 설정

In [ ]:
import os
import re
import time
import queue
import threading

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# 제스처 수학(디코딩·시퀀스·게이트)은 전부 gesture_live.py 에 있다. 여기서 다시 쓰지 않는다.
from gesture_live import (
    Runner, GestureClassifier, SequenceBuffer,
    build_sequence, is_active, decode, draw,
    GESTURE_CLASSES,
)

# ---- 카메라 ----
CAM_A_DEV = "/dev/cam0"      # 사용자 — 제스처를 본다
CAM_B_DEV = "/dev/cam1"      # 대상   — 두 번째 모델이 보고, 제스처로 화면을 조종한다
WIDTH, HEIGHT = 640, 480     # 00번 검증 스펙이자 제스처 학습 때의 프레임 비율
FOURCC = "MJPG"              # 고정. YUYV 는 이 카메라에서 30fps 가 안 나온다
TARGET_FPS = 30

DISPLAY_SCALE = 0.6
DISPLAY_FPS = 10
JPEG_QUALITY = 70
DISP_WH = (int(WIDTH * DISPLAY_SCALE), int(HEIGHT * DISPLAY_SCALE))   # 크롭해도 위젯이 안 튄다

# ---- 마이크 (01_wakeword 3장 값 그대로) ----
MIC_INDEX = 5
SAMPLE_RATE, CHANNELS, DTYPE, CHUNK = 16000, 1, "int16", 1280
WAKEWORD_MODEL = "hey_jarvis"     # 커스텀 hey_kria.onnx 가 나오면 경로로 교체
WAKE_THRESHOLD = 0.5
WAKE_REFRACTORY = 2.0

# ---- DPU ----
BIT_PATH = "dpu.bit"
GESTURE_XMODEL = "./gesture_stage1_kv260.xmodel"
TASK_XMODEL = GESTURE_XMODEL      # ← 데모용 대역. 진짜 2번 모델이 생기면 이 줄만 바꾼다
GRU_PATH = "./gesture_stage2_gru.tflite"

# ---- 제스처 (gesture_live.py 기본값) ----
CONF = 0.15                  # 학습 시퀀스 추출값. 바꾸면 특징 분포가 어긋난다
WINDOW_S = 1.0
GRU_EVERY = 5
GESTURE_THRES = 0.70
GESTURE_REFRACTORY = 1.5

# ---- cam1 디지털 PTZ ----
# 시연하면서 맞추는 값이다. 한 번에 얼마나 움직이고 얼마나 당길지는 손에 붙어야 정해진다.
PAN_STEP = 0.15              # 화면 폭 대비 1회 이동량 (줌 배율로 나눠 쓴다)
ZOOM_STEP = 1.25             # 1회 줌 배율
ZOOM_MIN, ZOOM_MAX = 1.0, 4.0
PAN_INVERT = False           # "left 제스처 = 화면이 왼쪽으로" 가 반대로 느껴지면 True


def dev_index(path):
    real = os.path.realpath(path)
    m = re.search(r"(\d+)$", real)
    if not m:
        raise ValueError(f"video 인덱스를 찾을 수 없다: {path} -> {real}")
    return int(m.group(1))


try:
    CAM_A_ID, CAM_B_ID = dev_index(CAM_A_DEV), dev_index(CAM_B_DEV)
except (ValueError, OSError) as e:
    print(f"[WARN] 심볼릭 링크 해석 실패 ({e}). 인덱스를 직접 지정한다.")
    CAM_A_ID, CAM_B_ID = 0, 2

print(f"CAM_A = {CAM_A_DEV} -> /dev/video{CAM_A_ID}  (제스처)")
print(f"CAM_B = {CAM_B_DEV} -> /dev/video{CAM_B_ID}  (두 번째 모델 · PTZ 대상)")
print(f"캡처 {WIDTH}x{HEIGHT} {FOURCC} @{TARGET_FPS}")
print(f"제스처 xmodel : {GESTURE_XMODEL}")
print(f"작업   xmodel : {TASK_XMODEL}"
      + ("   ← 같은 파일이다 (데모용 대역)" if TASK_XMODEL == GESTURE_XMODEL else ""))
print(f"PTZ           : pan {PAN_STEP} · zoom ×{ZOOM_STEP} ({ZOOM_MIN}~{ZOOM_MAX})")
print(f"웨이크 워드   : {WAKEWORD_MODEL}  threshold={WAKE_THRESHOLD}")

## 2. 캡처 스레드

`01_dual_camera_view.ipynb` 3장과 같다. `.ipynb` 는 import 가 안 되므로 옮겨 왔다.

카메라마다 스레드를 하나씩 둔다. 두 소비자(화면·DPU 워커)가 각자 `snapshot()` 으로
최신 프레임 한 장만 가져가므로 큐가 없고 지연이 쌓이지 않는다.

In [ ]:
def set_v4l2_ctrl(dev_id, name, value):
    import subprocess
    return subprocess.run(
        ["v4l2-ctl", "-d", f"/dev/video{dev_id}", "-c", f"{name}={value}"],
        capture_output=True, text=True,
    ).returncode == 0


class CamStream(threading.Thread):
    """카메라 한 대에서 계속 읽으며 최신 프레임과 실효 fps를 유지한다."""

    def __init__(self, dev_id, label):
        super().__init__(daemon=True)
        self.dev_id = dev_id
        self.label = label
        self.frame = None
        self.fps = 0.0
        self.reads = 0
        self.fails = 0
        self._lock = threading.Lock()
        self._running = False
        self.cap = None

    def open(self):
        cap = cv2.VideoCapture(self.dev_id, cv2.CAP_V4L2)
        if not cap.isOpened():
            cap.release()
            return False

        # FOURCC를 해상도보다 먼저 설정해야 한다
        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*FOURCC))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
        cap.set(cv2.CAP_PROP_FPS, TARGET_FPS)
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

        # 저조도에서 카메라가 프레임 주기를 2배로 늘리는 것을 막는다
        set_v4l2_ctrl(self.dev_id, "exposure_dynamic_framerate", 0)

        self.cap = cap
        return True

    def run(self):
        self._running = True
        n, t0 = 0, time.monotonic()
        while self._running:
            ok, f = self.cap.read()
            if ok and f is not None:
                with self._lock:
                    self.frame = f
                self.reads += 1
                n += 1
                if n >= 15:
                    now = time.monotonic()
                    self.fps = n / (now - t0)
                    n, t0 = 0, now
            else:
                self.fails += 1
                time.sleep(0.005)

    def snapshot(self):
        with self._lock:
            return None if self.frame is None else self.frame.copy()

    def stop(self):
        self._running = False
        self.join(timeout=2.0)
        if self.cap is not None:
            self.cap.release()
            self.cap = None

## 3. 상태 기계

파이프라인 전체의 규칙이 `apply_event()` 하나에 들어 있다. 하드웨어 없이 검증된다.

- 상태는 딕셔너리 하나다: `on`(화면 켜짐), `dpu`(`gesture`|`idle`|`task`),
  `cx`/`cy`/`zoom`(cam1 뷰포트)
- 웨이크 워드는 어느 상태에서든 켜고 `dpu=gesture` 로 되돌린다 — 항상 되돌아올 길이 있다
- 웨이크 워드와 제스처가 **같은 큐**로 들어와 워커 한 곳에서만 소비된다. 그래서 경합이 없다
- 함수가 새 딕셔너리를 돌려주고 워커가 통째로 대입하므로, 표시 루프가 반쪽짜리 상태를
  읽는 일이 없다. 락이 필요 없는 이유다
- 뷰포트(`cx`/`cy`/`zoom`)는 `dpu` 전환에 영향받지 않는다. `open` 으로 갔다 웨이크 워드로
  돌아와도 cam1 은 보던 자리를 그대로 보고 있다
- `_clamp()` 가 뷰포트를 프레임 안에 가둔다. 배율 1.0 에서는 여백이 0 이라 팬이 안 먹는데,
  버그가 아니라 잘라낼 것이 없어서다 — 먼저 `thumbs up` 으로 당겨야 한다

In [ ]:
INIT_STATE = {"on": False, "dpu": "gesture", "cx": 0.5, "cy": 0.5, "zoom": 1.0}
PAN = {"left": (-1, 0), "right": (1, 0), "up": (0, -1), "down": (0, 1)}
DPU_BY_EVENT = {"close": "idle", "open": "task"}     # dpu: gesture | idle | task


def _clamp(st):
    half = 0.5 / st["zoom"]                        # 뷰포트가 프레임 밖으로 못 나간다
    st["cx"] = min(1 - half, max(half, st["cx"]))
    st["cy"] = min(1 - half, max(half, st["cy"]))
    return st


def apply_event(st, ev):
    """상태 딕셔너리를 이벤트 하나로 갱신해 **새** 딕셔너리를 돌려준다. 순수 함수.

    낮은 배율에서는 한 스텝이 남은 여백보다 커서 한 번에 가장자리까지 간다.
    PAN_STEP 을 배율로 나누는 이유가 그것이고, 그래도 모자라면 PAN_STEP 을 줄인다.
    """
    st = dict(st)
    if ev == "wake":
        st["on"], st["dpu"] = True, "gesture"
        return st
    if not st["on"]:
        return st                                  # 꺼져 있으면 wake 말고는 아무것도 안 먹는다
    if ev == "bye":
        st["on"] = False
    elif ev in DPU_BY_EVENT:
        st["dpu"] = DPU_BY_EVENT[ev]
    elif ev in ("thumbs up", "thumbs down"):
        z = st["zoom"] * (ZOOM_STEP if ev == "thumbs up" else 1 / ZOOM_STEP)
        st["zoom"] = min(ZOOM_MAX, max(ZOOM_MIN, z))
    elif ev in PAN:
        dx, dy = PAN[ev]
        s = -PAN_STEP / st["zoom"] if PAN_INVERT else PAN_STEP / st["zoom"]
        st["cx"] += dx * s
        st["cy"] += dy * s
    return _clamp(st)


# --- 자체 점검 (하드웨어 불필요) ---
s = INIT_STATE
assert apply_event(s, "left") == s                          # 1) 꺼져 있으면 제스처 무시
assert apply_event(s, "bye") == s
assert apply_event(s, "open") == s                          #    꺼진 동안엔 모델도 안 바뀐다
s = apply_event(s, "wake")
assert s["on"] and s["dpu"] == "gesture"                    # 2) wake 로 켜지고 제스처 모델

s = apply_event(s, "thumbs up")
assert s["zoom"] > 1.0                                      # 3) 줌 인
s = apply_event(s, "left")
assert s["cx"] < 0.5                                        # 4) 줌 인 상태에서 팬이 먹는다
s = apply_event(s, "up")
assert s["cy"] < 0.5
zoomed = (s["cx"], s["cy"], s["zoom"])

assert apply_event(s, "close")["dpu"] == "idle"             # 5) close -> cam0 xmodel 끈다
s = apply_event(s, "open")
assert s["dpu"] == "task"                                   # 6) open  -> cam1 xmodel 켠다

# 7) 모델을 갈아 끼워도 cam1 뷰포트는 그대로 남는다
assert (s["cx"], s["cy"], s["zoom"]) == zoomed
s = apply_event(s, "wake")
assert s["dpu"] == "gesture"                                # 8) 복귀는 wake 뿐
assert (s["cx"], s["cy"], s["zoom"]) == zoomed

for _ in range(20):
    s = apply_event(s, "thumbs down")
assert s["zoom"] == ZOOM_MIN                                # 9) 줌 하한
assert (s["cx"], s["cy"]) == (0.5, 0.5)                     # 10) 줌 아웃하면 중앙으로 되돌아온다
assert apply_event(s, "left")["cx"] == 0.5                  # 11) 배율 1 에서는 이동 여백이 없다
for _ in range(20):
    s = apply_event(s, "thumbs up")
assert s["zoom"] == ZOOM_MAX                                # 12) 줌 상한

s = apply_event(s, "bye")
assert not s["on"]                                          # 13) bye 로 꺼진다
assert apply_event(s, "wake")["on"]
assert apply_event(s, "hi") == s                            # 14) hi 는 무동작
del s, zoomed
print("상태 기계 OK")

## 4. 웨이크 워드 리스너 (항상 켜져 있다)

`01_wakeword.ipynb` 7장의 `stream_detect()` 를 무한 실행 스레드로 바꾼 것이다.
콜백에서는 큐에 넣기만 하고 추론은 루프에서 한다 — 콜백 안에서 추론하면 오디오가 드롭된다.

이 스레드는 **DPU 를 전혀 만지지 않는다.** 그래서 xmodel 교체 중에도, 제스처 추론이
DPU 를 붙잡고 있어도 계속 듣는다.

In [ ]:
import sounddevice as sd
from openwakeword.model import Model


class WakeListener(threading.Thread):
    """마이크를 계속 듣다가 웨이크 워드가 잡히면 event_q 에 "wake" 를 넣는다."""

    def __init__(self, event_q):
        super().__init__(daemon=True)
        self.event_q = event_q
        self.q = queue.Queue()
        self.score = 0.0
        self.n_wake = 0
        self.drops = 0
        self.error = None
        self._running = False

    def _callback(self, indata, frames, time_info, status):
        if status:
            self.drops += 1
        self.q.put(indata[:, 0].copy())

    def run(self):
        self._running = True
        try:
            oww = Model(wakeword_models=[WAKEWORD_MODEL], inference_framework="onnx")
            key = list(oww.models.keys())[0]
            last_fire = -1e9
            with sd.InputStream(samplerate=SAMPLE_RATE, channels=CHANNELS, dtype=DTYPE,
                                blocksize=CHUNK, device=MIC_INDEX, callback=self._callback):
                print(f"[MIC ] 듣는 중 — \"{key}\"")
                while self._running:
                    try:
                        frame = self.q.get(timeout=0.5)
                    except queue.Empty:
                        continue
                    self.score = float(oww.predict(frame)[key])
                    now = time.monotonic()
                    if self.score > WAKE_THRESHOLD and (now - last_fire) > WAKE_REFRACTORY:
                        last_fire = now
                        self.n_wake += 1
                        print(f"[WAKE] score={self.score:.3f}")
                        self.event_q.put("wake")
        except Exception as e:
            self.error = f"{type(e).__name__}: {e}"
            print(f"[MIC ] 죽었다 — {self.error}")

    def stop(self):
        self._running = False
        self.join(timeout=3.0)

## 5. DPU 워커 (하나뿐인 DPU 를 독점한다)

모드 전환 = `rt.load_model()`. 비트스트림은 `Runner()` 생성 때 한 번만 올라간다.
`idle` 에서는 아무 모델도 올리지 않고 DPU 를 놀린다.

교체할 때 **시퀀스 버퍼를 반드시 비운다.** 안 비우면 교체 직전 1초 창이 살아남아
제스처 모드로 돌아오는 순간 그 제스처가 다시 발화하고 곧바로 튕겨 나간다.

`TASK_XMODEL` 이 대역이라 제스처 xmodel 과 같은 파일이어도 **일부러 다시 올린다.**
교체 비용(`[SWAP]` 의 ms)이 이 데모에서 재려는 값이기 때문이다.

`SequenceBuffer` 는 프레임 수가 아니라 시간 창(`window_s`)으로 30점을 리샘플링하므로,
이 루프가 카메라보다 느려도 GRU 가 보는 속도 특징은 유지된다. 손댈 곳이 없다.

In [ ]:
class DpuWorker(threading.Thread):
    """상태에 따라 xmodel 을 갈아 끼우고, 인식된 제스처를 이벤트 큐로 보낸다."""

    def __init__(self, event_q, cams):
        super().__init__(daemon=True)
        self.event_q = event_q
        self.cams = cams            # [cam0(제스처), cam1(작업 모델 · PTZ 대상)]
        self.state = INIT_STATE     # 통째로 교체만 한다 — 표시 루프가 반쪽 상태를 볼 일이 없다
        self.loaded = os.path.basename(GESTURE_XMODEL)
        self.det = None             # {"cam": i, "boxes"/"scores"/"cls_ids"}
        self.dpu_ms = 0.0
        self.swap_ms = 0.0
        self.gesture = ""
        self.error = None
        self.ready = False
        self._running = False

    def _drain(self, st):
        while True:
            try:
                ev = self.event_q.get_nowait()
            except queue.Empty:
                return st
            new = apply_event(st, ev)
            if new != st:
                print(f"[MODE] {ev:12s} -> on={new['on']} dpu={new['dpu']} "
                      f"zoom={new['zoom']:.2f} center=({new['cx']:.2f}, {new['cy']:.2f})")
            st = new

    def run(self):
        self._running = True

        # PYNQ 는 인터럽트를 asyncio 로 다뤄서 오버레이를 만질 때 이벤트 루프를 요구한다.
        # 워커 스레드에는 루프가 없고, 3.10+ 부터는 자동 생성도 안 해준다:
        #   RuntimeError: There is no current event loop in thread 'Thread-N'
        # 교체(load_model)도 이 스레드에서 도니 생성만 메인 스레드로 옮겨서는 못 고친다.
        import asyncio
        asyncio.set_event_loop(asyncio.new_event_loop())

        try:
            rt = Runner(BIT_PATH, GESTURE_XMODEL)
            gru = GestureClassifier(GRU_PATH)
            print(f"[DPU ] input={rt.in_dims} layout={rt.layout} int8={rt.in_is_int8}")
        except Exception as e:
            self.error = f"{type(e).__name__}: {e}"
            print(f"[DPU ] 초기화 실패 — {self.error}")
            return

        seqbuf = SequenceBuffer(window_s=WINDOW_S)
        st = INIT_STATE
        run_mode = "off"             # 직전 루프가 실제로 돌던 모드
        n, last_fire = 0, -1e9
        self.ready = True

        while self._running:
            st = self._drain(st)
            self.state = st
            mode = "off" if not st["on"] else st["dpu"]     # off | idle | gesture | task

            if mode != run_mode:
                # gesture/task 로 들어갈 때만 DPU 에 모델을 올린다. idle 은 아무것도 안 올린다.
                # TASK_XMODEL 이 대역(같은 파일)이어도 일부러 다시 올린다 — 교체 비용을 재려고.
                if mode in ("gesture", "task"):
                    want = GESTURE_XMODEL if mode == "gesture" else TASK_XMODEL
                    t0 = time.perf_counter()
                    rt.load_model(want)
                    self.swap_ms = (time.perf_counter() - t0) * 1000
                    self.loaded = os.path.basename(want)
                    print(f"[SWAP] {mode} · {self.loaded} 로 교체  {self.swap_ms:.0f} ms")
                # 교체 전 1초 창을 버린다. 안 버리면 제스처 모드로 돌아오는 순간 직전
                # 제스처가 그대로 다시 발화한다.
                seqbuf.buf.clear()
                last_fire = time.time()
                self.det = None
                self.gesture = ""
                run_mode = mode

            if mode in ("off", "idle"):     # DPU 를 아예 쓰지 않는다
                time.sleep(0.05)
                continue

            idx = 0 if mode == "gesture" else 1
            frame = self.cams[idx].snapshot()
            if frame is None:
                time.sleep(0.01)
                continue

            inp, ratio, pad, orig_wh = rt.preprocess(frame)
            t0 = time.perf_counter()
            outputs = rt.infer(inp)
            boxes, scores, cls_ids = decode(outputs, ratio, pad, orig_wh, CONF)
            self.dpu_ms = (time.perf_counter() - t0) * 1000
            self.det = {"cam": idx, "boxes": boxes, "scores": scores, "cls_ids": cls_ids}

            if mode == "task":
                continue             # 작업 모델은 그리기만 한다 (대역이라 후처리가 같다)

            # ---- 2단계: 특징 시퀀스 -> GRU (gesture_live.main 과 같은 규칙) ----
            now = time.time()
            if len(boxes):
                x1, y1, x2, y2 = boxes[0]
                ow, oh = orig_wh
                det = ((x1 + x2) / 2 / ow, (y1 + y2) / 2 / oh,
                       (x2 - x1) / ow, (y2 - y1) / oh,
                       float(scores[0]), int(cls_ids[0]))
            else:
                det = None
            seqbuf.push(now, det)

            n += 1
            if n % GRU_EVERY or not seqbuf.ready(now):
                continue

            seq = build_sequence(seqbuf.sample(now))
            if seq is None:
                continue

            active, disp, shape = is_active(seq)
            if not active:                       # 정지 게이트: 손이 멈춰 있으면 GRU 를 안 부른다
                self.gesture = ""
                continue

            prob = gru.predict(seq)
            k = int(np.argmax(prob))
            if prob[k] < GESTURE_THRES or (now - last_fire) < GESTURE_REFRACTORY:
                continue

            last_fire = now
            self.gesture = f"{GESTURE_CLASSES[k]} {prob[k]:.2f}"
            print(f"[GEST] {GESTURE_CLASSES[k]:12s} p={prob[k]:.3f}")
            # 상태를 직접 건드리지 않고 wake 와 같은 큐로 보낸다. 소비자는 _drain 한 곳뿐이다.
            self.event_q.put(GESTURE_CLASSES[k])

    def stop(self):
        self._running = False
        self.join(timeout=5.0)

## 6. 실행

셀을 실행하면 화면 두 개가 뜬다. **Stop 버튼으로 종료한다.** 셀을 중단(interrupt)하지 말 것.

테두리가 지금 DPU 가 보고 있는 카메라다 — 초록 `gesture`, 파랑 `task`.

1. 시작 직후 — `OFF`. 두 화면 정지. DPU 유휴
2. 웨이크 워드 → `GESTURE`. cam0 에 손 박스가 뜬다
3. `thumbs up` → cam1 이 확대된다. 그 다음 `left`/`right`/`up`/`down` 으로 옮긴다
   (배율 1.0 에서는 옮길 여백이 없다. 줌부터 하는 게 순서다)
4. `open` → `[SWAP]` 이 찍히고 박스가 cam1 로 넘어간다 (`TASK`)
5. 다시 웨이크 워드 → cam0 으로 복귀. **cam1 은 보던 자리·배율 그대로다**
6. `close` → `IDLE`. 두 화면 다 박스 없음, DPU 유휴. 복귀는 웨이크 워드
7. `bye` → 두 화면 정지. 복귀는 웨이크 워드

3~7 은 전부 제스처 모드에서만 낼 수 있다. `TASK`·`IDLE`·`OFF` 에서는 cam0 의 제스처
xmodel 이 내려가 있어 손을 아무리 흔들어도 안 잡힌다. 말로 돌아와야 한다.

`Wake (수동)` 버튼은 마이크가 말을 안 들을 때 같은 이벤트를 직접 넣는다. 시연용 보험이다.
제스처도 손 없이 넣어 보고 싶으면 `event_q.put("open")` 을 셀에서 직접 부르면 된다.

In [ ]:
streams, view, worker, listener = [], None, None, None
event_q = queue.Queue()

STATE_COLOR = {"off": (120, 120, 120), "idle": (60, 60, 200),
               "gesture": (0, 220, 120), "task": (0, 180, 255)}


def state_label(st):
    return "off" if not st["on"] else st["dpu"]


def crop_view(frame, st):
    """cam1 디지털 PTZ. zoom=1.0 이면 원본 그대로 돌려준다."""
    if st["zoom"] <= 1.0:
        return frame
    h, w = frame.shape[:2]
    cw, ch = int(w / st["zoom"]), int(h / st["zoom"])
    x = min(w - cw, max(0, int(st["cx"] * w - cw / 2)))
    y = min(h - ch, max(0, int(st["cy"] * h - ch / 2)))
    return frame[y:y + ch, x:x + cw]


def overlay(frame, text, active, color):
    """좌상단 라벨 + 활성 카메라 테두리."""
    out = frame.copy()
    cv2.rectangle(out, (0, 0), (330, 34), (0, 0, 0), -1)
    out = cv2.addWeighted(out, 0.65, frame, 0.35, 0)
    cv2.putText(out, text, (8, 24), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (0, 255, 120), 1, cv2.LINE_AA)
    if active:
        h, w = out.shape[:2]
        cv2.rectangle(out, (0, 0), (w - 1, h - 1), color, 4)
    return out


def start():
    global streams, view, worker, listener

    stop_all()
    while not event_q.empty():
        event_q.get_nowait()

    streams = [CamStream(CAM_A_ID, "CAM_A 사용자"), CamStream(CAM_B_ID, "CAM_B 대상")]
    for s in streams:
        if not s.open():
            print(f"[FAIL] {s.label} (/dev/video{s.dev_id}) 열기 실패")
            print("       dmesg | grep -i 'Not enough bandwidth' 로 확인할 것")
            stop_all()
            return
        s.start()
    time.sleep(0.5)

    listener = WakeListener(event_q)
    listener.start()
    worker = DpuWorker(event_q, streams)
    worker.start()

    imgs = [widgets.Image(format="jpeg") for _ in streams]
    caps = [widgets.HTML(f"<b>{s.label}</b>") for s in streams]
    cols = [widgets.VBox([c, i]) for c, i in zip(caps, imgs)]

    btn_stop = widgets.Button(description="Stop", button_style="danger", icon="stop")
    btn_wake = widgets.Button(description="Wake (수동)", button_style="info")
    status = widgets.HTML("시작 중… (DPU 비트스트림 로드에 몇 초 걸린다)")

    view = {"running": True}
    btn_wake.on_click(lambda _: event_q.put("wake"))

    def on_stop(_):
        view["running"] = False
        btn_stop.disabled = True
        btn_stop.description = "Stopped"

    btn_stop.on_click(on_stop)
    display(widgets.VBox([widgets.HBox(cols), widgets.HBox([btn_stop, btn_wake, status])]))

    def loop(state):
        period = 1.0 / DISPLAY_FPS
        while state["running"]:
            t0 = time.monotonic()
            st = worker.state              # 딕셔너리 통째 교체라 스냅샷 한 번이면 된다
            label = state_label(st)
            color = STATE_COLOR[label]
            det = worker.det

            if st["on"]:                   # 꺼져 있으면 위젯을 갱신하지 않는다 -> 화면이 얼어붙는다
                for i, (w, s) in enumerate(zip(imgs, streams)):
                    f = s.snapshot()
                    if f is None:
                        continue
                    if det is not None and det["cam"] == i:
                        draw(f, det["boxes"], det["scores"], det["cls_ids"])  # 크롭·축소 전에
                    if i == 1:
                        f = crop_view(f, st)
                    f = cv2.resize(f, DISP_WH, interpolation=cv2.INTER_AREA)
                    tag = (f"{s.label}  {s.fps:4.1f} fps" if i == 0
                           else f"{s.label}  x{st['zoom']:.2f}")
                    f = overlay(f, tag, det is not None and det["cam"] == i, color)
                    ok, buf = cv2.imencode(".jpg", f, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
                    if ok:
                        w.value = buf.tobytes()

            status.value = (
                f"&nbsp;&nbsp;<b>{label.upper()}</b> · {worker.loaded} · "
                f"DPU {worker.dpu_ms:.0f}ms · swap {worker.swap_ms:.0f}ms · "
                f"zoom x{st['zoom']:.2f} 중심 ({st['cx']:.2f}, {st['cy']:.2f}) · "
                f"제스처 {worker.gesture or '—'} · "
                f"wake {listener.n_wake}회 (score {listener.score:.2f}, drop {listener.drops})"
                + (f" · <span style='color:crimson'>{worker.error or listener.error}</span>"
                   if (worker.error or listener.error) else "")
            )
            time.sleep(max(0.0, period - (time.monotonic() - t0)))

        stop_all()
        status.value = "&nbsp;&nbsp;정지됨. 카메라·마이크·DPU 를 해제했다."

    threading.Thread(target=loop, args=(view,), daemon=True).start()


def stop_all():
    global streams, view, worker, listener
    if view is not None:
        view["running"] = False
        view = None
    for obj in [listener, worker, *streams]:
        try:
            obj.stop()
        except Exception:
            pass
    listener, worker, streams = None, None, []

In [ ]:
start()

## 7. 정리

셀을 중단했거나 커널을 재시작하기 전에 실행한다.
카메라나 마이크가 열린 채 남으면 다음 실행에서 열기가 실패한다.

In [ ]:
stop_all()
print("카메라·마이크·DPU 해제 완료")

## 8. 문제 해결

### 처음에 화면이 안 나온다

정상이다. `on=False` 로 시작하므로 웨이크 워드(또는 `Wake (수동)`) 전에는 위젯을
갱신하지 않는다. 마이크가 안 잡히면 버튼을 쓴다.

### `open`/`close` 한 뒤로 제스처가 하나도 안 먹는다

정상이다. DPU 가 하나라 cam0 의 제스처 xmodel 이 내려가 있다. 상태줄이 `TASK` 나
`IDLE` 이면 손은 아무 의미가 없다 — 웨이크 워드로 `GESTURE` 로 돌아와야 한다.

### `[MODE]` 는 찍히는데 `[SWAP]` 이 안 찍힌다

`gesture -> idle` 이나 `-> off` 는 교체가 아니다. 올릴 모델이 없으니 건너뛴다.
`[SWAP]` 은 `gesture`/`task` 로 **들어갈 때만** 찍힌다.

### `[GEST]` 는 찍히는데 cam1 이 안 움직인다

상태줄을 먼저 본다.

- `zoom x1.00` 이다 → 잘라낼 여백이 없다. `thumbs up` 부터 한다
- 중심이 이미 가장자리다 (`0.20` 또는 `0.80` 같은 값에 붙어 멈춤) → 더 갈 곳이 없다

### 움직이는 방향이 반대다

`PAN_INVERT = True`. 사람마다 "왼쪽으로"가 카메라 기준인지 화면 기준인지 다르다.

### 한 번에 너무 많이/적게 움직인다

`PAN_STEP`(기본 0.15), `ZOOM_STEP`(기본 1.25)을 조절한다. 낮은 배율에서는 한 스텝이
남은 여백보다 커서 한 번에 끝까지 가는데, 그게 거슬리면 `PAN_STEP` 을 0.08 정도로 내린다.

### 오디오 드롭(`drop`)이 0 이 아니다

DPU 추론과 카메라 디코딩이 A53 4코어를 나눠 쓰면서 오디오 스레드가 굶은 것이다.
순서대로 내린다.

```python
DISPLAY_FPS = 6        # 먼저 이것
DISPLAY_SCALE = 0.4
GRU_EVERY = 8          # 그래도 안 되면
```

01번에서 잰 `cpu_load_pct` 와 비교한다. 웨이크 워드 예산은 청크당 80 ms 다.

### 두 번째 카메라가 안 열린다

```bash
dmesg | grep -iE "Not enough bandwidth|Capping" | tail
sudo /usr/local/sbin/uvc-swap.sh on 1024
```

### 제스처가 안 잡힌다

`gesture_live.py --gate-debug` 로 단독 확인하는 편이 빠르다. 정지 게이트(`is_active`)가
막고 있으면 손을 크게 움직이거나(`disp > 0.04`) 손모양을 바꾼다(`shape > 0.5`).

박스 자체가 안 뜨면 게이트가 아니라 1단계 문제다. 카메라 거리와 조명을 본다.

### 교체 직후 제스처가 한 번 더 발화한다

`seqbuf.buf.clear()` 가 빠진 경우다. 5장 워커의 전환 블록에 이미 들어 있으니 지우지 말 것.

### 교체가 느리다

`[SWAP]` 의 ms 를 본다. 비트스트림까지 다시 올리면 초 단위가 되지만 여기서는
`overlay.load_model()` 만 부른다. 초 단위가 나온다면 `Runner` 를 새로 만들고 있는 것이다.

---

### 다음 단계

- `TASK_XMODEL` 을 진짜 두 번째 모델로 바꾸고, 그 모델의 후처리를 워커의 `mode == "task"`
  분기에 넣는다 (지금은 대역이라 제스처 `decode()` 를 그대로 쓴다).
- cam1 이 진짜 PTZ 카메라로 바뀌면 `crop_view()` 자리에 `set_v4l2_ctrl(..., "pan_absolute", ...)`
  를 넣는다. 상태와 매핑은 그대로 쓴다. C270 에는 그 컨트롤이 없어 지금은 디지털 크롭이다.
- 커스텀 `hey_kria.onnx` 가 나오면 `WAKEWORD_MODEL` 을 경로로 바꾼다.